In [1]:
import os, sys

sys.path.append(
    os.path.dirname(os.getcwd())
)
import json
from pprint import pprint
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2

import trimesh
import plotly.graph_objects as go
import PIL
from PIL import Image

import pygarment as pyg

from analysis_utils import (
    set_constants,
    # visualize_meshes_plotly,
    euler_angles_to_rotation_matrix,
    plot_panel_info,
    v_id_map
)


DATASET_ROOT = set_constants()

GARMENT_ROOT_PATH = os.path.join(DATASET_ROOT, "GarmentCodeData_v2")
BODY_ROOT_PATH = os.path.join(DATASET_ROOT, "body_mesh")
MEAN_ALL_BODY_PATH = os.path.join(DATASET_ROOT, "neutral_body/mean_all.obj")

default_body_mesh = trimesh.load(MEAN_ALL_BODY_PATH)


# BODY_TYPE = "random_body"
BODY_TYPE = "default_body"

garment_path_list = sorted(list(filter(
    os.path.isdir,
    glob(os.path.join(GARMENT_ROOT_PATH, "*", BODY_TYPE, "*"))
)))

len(garment_path_list)

132670

In [2]:
import json
#TODO: 내가 쓴 코드.
json_file_path = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/garment_without_arc_list.json"

def load_json_file(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

json_data = load_json_file(json_file_path)
garment_path_list = json_data

In [4]:
json_data


['/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/GarmentCodeData_v2/garments_5000_14/default_body/rand_8SO8WAA1A6',
 '/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/GarmentCodeData_v2/garments_5000_17/default_body/rand_0RFKFCW8RL',
 '/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/GarmentCodeData_v2/garments_5000_5/default_body/rand_0X3G7BGJA6',
 '/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/GarmentCodeData_v2/garments_5000_12/default_body/rand_XYPIS8CW1U',
 '/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/GarmentCodeData_v2/garments_5000_35/default_body/rand_QE1UCI1XEG',
 '/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/GarmentCodeData_v2/garments_5000_0/default_body/rand_ZL3KYW7IB5',
 '/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/GarmentCodeData_v2/garments_5000_19/default_body/rand_YHGKB7O1KY',
 '/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/Garme

In [3]:
def visualize_meshes_plotly(
    mesh_list,
    color_list=None,
    vertices_list = None,
    vertices_color_list = None,
    vertex_marker_size = 2,
    show_edges = True,
    edge_width = 2,
):
    # Pre-convert to list and load meshes once
    mesh_list = [mesh_list] if not isinstance(mesh_list, list) else mesh_list
    final_mesh_list = [trimesh.load(m) if isinstance(m, str) else m for m in mesh_list]
    
    color_list = color_list or ['lightgray'] * len(final_mesh_list)
    
    # Create all mesh traces at once
    mesh_traces = []
    edge_traces = []
    for mesh, color in zip(final_mesh_list, color_list):
        face_colors = mesh.visual.face_colors[:, :3] if hasattr(mesh.visual, 'face_colors') else None
        mesh_traces.append(go.Mesh3d(
            x=mesh.vertices[:, 0],
            y=mesh.vertices[:, 1],
            z=mesh.vertices[:, 2],
            i=mesh.faces[:, 0],
            j=mesh.faces[:, 1],
            k=mesh.faces[:, 2],
            opacity=0.5,
            facecolor=face_colors,
            color=None if face_colors is not None else color
        ))
        
        if show_edges:
            edge_x = []
            edge_y = []
            edge_z = []
            vertices = mesh.vertices
            for edge in mesh.edges:
                edge_x.extend([vertices[edge[0], 0], vertices[edge[1], 0], None])
                edge_y.extend([vertices[edge[0], 1], vertices[edge[1], 1], None])
                edge_z.extend([vertices[edge[0], 2], vertices[edge[1], 2], None])
        
            edge_traces.append(go.Scatter3d(
                x=edge_x, y=edge_y, z=edge_z,
                mode='lines',
                line=dict(
                    color = color if color is not None else 'red',
                    width=edge_width
                ),
                name='Edges'
            ))
    
    fig = go.Figure(data = mesh_traces + edge_traces)
    
    if vertices_list is not None and vertices_color_list is not None:
        for vertex, color in zip(vertices_list, vertices_color_list):
            fig.add_trace(go.Scatter3d(
                x=vertex[:, 0],
                y=vertex[:, 1],
                z=vertex[:, 2],
                mode='markers',
                marker=dict(size=vertex_marker_size, color=color, opacity=1),
                name='Vertices'
            ))
            fig.add_trace(go.Scatter3d(
                x=vertex[:, 0],
                y=vertex[:, 1],
                z=vertex[:, 2],
                mode='lines',
                line=dict(color=color, width=2),
                name='Polyline'
            ))
    fig.update_layout(
        scene=dict(aspectmode='data'),
        width=800,
        height=800,
        showlegend=False
    )
    
    fig.show()


In [7]:
# ARC가 아닌 데이터에 대해 각 panel의 center point와 해당 패널 이름을 각각 저장.



for IDX in len(garment_path_list):
    garment_path = garment_path_list[IDX]
    garment_id = os.path.basename(garment_path)
    SPEC_FILE_PATH = os.path.join(garment_path, f"{garment_id}_specification.json")

    with open(SPEC_FILE_PATH, "r") as f:
        spec = json.load(f)
    
    pattern = pyg.pattern.wrappers.VisPattern(SPEC_FILE_PATH)

    # Get Garment Blueprint
    panel_svg_path_dict = {
        panel_name : pattern._draw_a_panel(
            panel_name, apply_transform=False, fill=True
        )
        for panel_name in pattern.panel_order()
    }
    
    lifted_panel_vertex_arr_list = []
    batch_point_list = []

    for panel_name, panel in panel_svg_path_dict.items():
        vertices_from_spec = np.asarray(spec['pattern']['panels'][panel_name]['vertices'])    
        offset = np.min(vertices_from_spec * np.array([1, -1]), axis=0)

        euler_angles = spec['pattern']['panels'][panel_name]['rotation']
        translation = spec['pattern']['panels'][panel_name]['translation']
        rotation_matrix = euler_angles_to_rotation_matrix(euler_angles)

        path = panel[0]
        
        x1, x2, y1, y2 = path.bbox()
        xm = (x1 + x2) / 2
        ym = (y1 + y2) / 2 

        vertices_from_svg = np.array(list(map(
            lambda edge : [edge.start.real, edge.start.imag],
            path
        )))
        vertices = np.vstack([
            vertices_from_svg,
            vertices_from_svg[0], # add start point to close the loop
        ])
        vertices = (vertices + offset) * np.array([1, -1])
        panel_vertex_arr = np.hstack([
            vertices,
            np.zeros((vertices.shape[0], 1))
        ]) @ rotation_matrix.T + translation
        
        lifted_panel_vertex_arr_list.append(panel_vertex_arr)
        
        bbox_center = np.array([xm, ym])
        bbox_center = (bbox_center + offset) * np.array([1, -1])
        
        batch_point = np.array([
            bbox_center[0], bbox_center[1], 1
        ]) @ rotation_matrix.T + translation
        
        batch_point_list.append(batch_point.reshape(1, -1))
        



TypeError: 'int' object is not iterable

In [19]:
import os
import csv
import json
import numpy as np
import tqdm
# 중심점을 저장할 전역 리스트
center_points_global = []

for garment_path in tqdm.tqdm(garment_path_list):
    garment_id = os.path.basename(garment_path)
    SPEC_FILE_PATH = os.path.join(garment_path, f"{garment_id}_specification.json")

    with open(SPEC_FILE_PATH, "r") as f:
        spec = json.load(f)
    
    # 패턴 객체 준비 (예: pyg.pattern.wrappers.VisPattern)
    pattern = pyg.pattern.wrappers.VisPattern(SPEC_FILE_PATH)

    # 패널별 svg path
    panel_svg_path_dict = {
        panel_name : pattern._draw_a_panel(panel_name, apply_transform=False, fill=True)
        for panel_name in pattern.panel_order()
    }

    for panel_name, panel in panel_svg_path_dict.items():
        # (1) JSON에서 회전, 평행이동 정보 읽어오기
        euler_angles = spec['pattern']['panels'][panel_name]['rotation']
        translation = spec['pattern']['panels'][panel_name]['translation']
        rotation_matrix = euler_angles_to_rotation_matrix(euler_angles)

        # (2) 좌표 오프셋 계산
        vertices_from_spec = np.asarray(spec['pattern']['panels'][panel_name]['vertices'])
        offset = np.min(vertices_from_spec * np.array([1, -1]), axis=0)

        # (3) 패널의 SVG path
        path = panel[0]
        
        # (4) 바운딩 박스 중심 계산
        x1, x2, y1, y2 = path.bbox()
        xm = (x1 + x2) / 2
        ym = (y1 + y2) / 2 

        bbox_center = np.array([xm, ym])
        bbox_center = (bbox_center + offset) * np.array([1, -1])

        # (5) 회전/이동 적용 (z=0 가정)
        batch_point = np.array([bbox_center[0], bbox_center[1], 0]) @ rotation_matrix.T + translation
        
        # (6) 전역 리스트에 저장
        center_points_global.append({
            "garment_id": garment_id,
            "panel_name": panel_name,
            "x": float(batch_point[0]),
            "y": float(batch_point[1]),
            "z": float(batch_point[2])
        })

# ----------------------------------
# 모든 garment에 대한 패널 중심점 계산이 끝났다면,
# center_points_global 리스트를 한 번에 CSV 파일로 저장
# ----------------------------------

output_csv = "all_garments_panel_centers.csv"
with open(output_csv, 'w', newline='') as f:
    # 컬럼(필드) 이름 정의
    fieldnames = ["garment_id", "panel_name", "x", "y", "z"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    # 헤더 작성
    writer.writeheader()
    
    # 리스트 데이터를 한 번에 기록
    writer.writerows(center_points_global)

print(f"[완료] 모든 의류에 대한 패널 중심점이 '{output_csv}'에 저장되었습니다!")

  0%|          | 0/30427 [00:00<?, ?it/s]

100%|██████████| 30427/30427 [01:08<00:00, 442.53it/s]


[완료] 모든 의류에 대한 패널 중심점이 'all_garments_panel_centers.csv'에 저장되었습니다!


In [11]:
import pandas as pd

# 1) CSV 파일 읽어오기
df = pd.read_csv("all_garments_panel_centers.csv")
# CSV 파일에 [garment_id, panel_name, x, y, z] 컬럼이 있다고 가정

# 2) panel_name 기준으로 그룹핑
grouped = df.groupby("panel_name")

# panel name unique
# grouped.panel_name.unique()

# panel name save list
panel_name_list = grouped.panel_name.unique()


[array(['ins_skirt_back_0'], dtype=object),
 array(['ins_skirt_back_1'], dtype=object),
 array(['ins_skirt_back_2'], dtype=object),
 array(['ins_skirt_back_3'], dtype=object),
 array(['ins_skirt_back_4'], dtype=object),
 array(['ins_skirt_back_5'], dtype=object),
 array(['ins_skirt_front_0'], dtype=object),
 array(['ins_skirt_front_1'], dtype=object),
 array(['ins_skirt_front_2'], dtype=object),
 array(['ins_skirt_front_3'], dtype=object),
 array(['ins_skirt_front_4'], dtype=object),
 array(['ins_skirt_front_5'], dtype=object),
 array(['left_btorso'], dtype=object),
 array(['left_ftorso'], dtype=object),
 array(['left_sleeve_b'], dtype=object),
 array(['left_sleeve_f'], dtype=object),
 array(['pant_b_l'], dtype=object),
 array(['pant_b_r'], dtype=object),
 array(['pant_f_l'], dtype=object),
 array(['pant_f_r'], dtype=object),
 array(['pant_l_cuff_b'], dtype=object),
 array(['pant_l_cuff_f'], dtype=object),
 array(['pant_l_cuff_skirt_b'], dtype=object),
 array(['pant_l_cuff_skirt_f'], d

In [14]:
grouped.get_group("skirt_back_2")

,garment_id,panel_name,x,y,z
676,rand_XYZJO8NRMR,skirt_back_2,0.000000,20.168721,-20.0
777,rand_EWLQ4A9U0U,skirt_back_2,0.411850,27.282105,-20.0
887,rand_KXB3AGNP3T,skirt_back_2,0.411850,33.350026,-20.0
951,rand_MRP56VAC5D,skirt_back_2,0.411850,19.416176,-20.0
1241,rand_AU0T3JE5SK,skirt_back_2,0.000000,24.437778,-20.0
...,...,...,...,...,...
198847,rand_T3D8MQC70K,skirt_back_2,0.411850,41.272500,-20.0
199203,rand_8VRP25DYCP,skirt_back_2,0.411850,29.098426,-20.0
199729,rand_KXQMRHA2AK,skirt_back_2,0.411850,42.960140,-20.0
199965,rand_8NI150ODPU,skirt_back_2,-0.056279,-11.603999,-20.0


SyntaxError: invalid syntax (3062011679.py, line 1)

In [8]:
panel_svg_path_dict

{'pant_b_r': (Path(QuadraticBezier(start=63.706243671475406j, control=(0.2053228758273652+32.5892256120269j), end=(0.20532195007181997+22.216718897386826j)),
       QuadraticBezier(start=(0.20532195007181997+22.216718897386826j), control=(0.20532212669128602+11.069374799577934j), end=(2.1226404209266008+0j)),
       Line(start=(2.1226404209266008+0j), end=(11.391972781642544+0j)),
       Line(start=(11.391972781642544+0j), end=(12.59917885575646+15.950419197970326j)),
       Line(start=(12.59917885575646+15.950419197970326j), end=(13.80638492987038+0j)),
       Line(start=(13.80638492987038+0j), end=(18.36493482290716+0j)),
       Line(start=(18.36493482290716+0j), end=(19.572140897021075+17.73232969963467j)),
       Line(start=(19.572140897021075+17.73232969963467j), end=(20.779346971134995+0j)),
       Line(start=(20.779346971134995+0j), end=(27.61717195007182+0j)),
       Line(start=(27.61717195007182+0j), end=(27.61717195007182+12.219195393562757j)),
       QuadraticBezier(start=(2

In [ ]:
77086
114120

In [1]:
print("asd")

asd


In [ ]:

SPEC_FILE_PATH = os.path.join(garment_path, f"{garment_id}_specification.json")
pattern = pyg.pattern.wrappers.VisPattern(SPEC_FILE_PATH)

# Get Garment Blueprint
panel_svg_path_dict = {
    panel_name : pattern._draw_a_panel(
        panel_name, apply_transform=False, fill=True
    )
    for panel_name in pattern.panel_order()
}
stitch_dict = {
    i : v for i, v in enumerate(pattern.pattern['stitches'])
}

FIGLEN = 4
NCOLS = int(np.sqrt(len(panel_svg_path_dict)))
NROWS = int(np.ceil(len(panel_svg_path_dict) / NCOLS))

plt.figure(figsize=(FIGLEN * NCOLS, FIGLEN * NROWS))
for i, (panel_name, panel) in enumerate(panel_svg_path_dict.items()):
    ax = plt.subplot(NROWS, NCOLS, i + 1)
    ax.set_title(panel_name)
    plot_panel_info(
        ax, panel_name, panel_svg_path_dict, stitch_dict,
        N_SAMPLES=1000
    )
# plt.savefig(f"{garment_id}_panel_vis.png")
plt.show()